# Feature selection and year-controlled benchmark models

**Purpose.** This notebook retains the frozen leakage policy, correlations and validation design, then reproduces the final supervisor-requested entry-year-controlled F1 and F3 models.

**Principal inputs.** Clean Y52 cohort; frozen protected folds; frozen feature roles; historical benchmark evidence; `supervisor_confirmed_update_v1`.

**Main outputs.** Predictor audit, correlation/VIF evidence, validation checks and reconciled F1_YEAR/F3_YEAR held-out metrics. Entry year is a pre-specified control, not a performance-selected feature.


In [ ]:
from pathlib import Path
import json
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

# Find the repository from the current working directory so the notebook
# remains portable when the project folder is moved or opened in Colab.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
ROOT = next((p for p in candidates if (p / "scripts/thesis_utils_v2.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the Spotify_clv project.")
sys.path.insert(0, str(ROOT / "scripts"))

from thesis_utils_v2 import (
    SEED, assert_close, assert_unique_tracks, blocked_predictor_hits,
    concise_table, environment_table, extract_spotify_id, load_json,
    read_chart_archive, record_values, regression_metrics,
    save_and_show_figure, setup_notebook,
)

ROOT, FIGURE_DIR, RECORD_DIR = setup_notebook("03_feature_selection_and_benchmarks.ipynb")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
display(environment_table())
print(f"Random seed: {SEED}")


## Predictor timing and feature-selection logic

A variable is admissible only if it is available by the end of calendar week 4 and is not derived from the later outcome history. Selection then proceeds through conceptually defined blocks and grouped out-of-sample comparison. No predictor was deleted automatically because of a p-value.


In [ ]:
roles = pd.read_csv(ROOT / "data/processed/final_pre_model_specification_v1/provisional_variable_roles.csv")
feature_record = pd.read_csv(ROOT / "data/processed/provisional_final_model_comparison_v1/final_feature_selection_record.csv")
role_counts = roles.groupby(["variable_role", "leakage_status"], dropna=False).size().reset_index(name="variables")
display(role_counts)
display(feature_record[[
    "variable_or_group", "final_status", "final_model_scope", "treatment", "reason", "leakage_status"
]].head(25))

blocked = feature_record[feature_record.final_status.str.contains("excluded|blocked", case=False, na=False)]
display(blocked[["variable_or_group", "final_status", "reason"]].head(20))


Blocked fields include popularity/follower snapshots with uncertain timing, future ranks or streams, eventual peak rank, total or first-spell duration, L/Z/re-entry summaries, and old landmark eligibility flags. They remain available only for audit or outcome description. Artist ID is a grouping key, not a numeric song predictor.

## Correlation and multicollinearity

Pearson and Spearman correlations are recomputed for the retained continuous/binary F3 and LEX variables. X and musical key are categorical in the models and are therefore omitted from this continuous-variable heatmap.


In [ ]:
data = pd.read_csv(ROOT / "data/processed/pre_model_analysis_ready_v1/analysis_ready_550k_52w.csv")
lexical = pd.read_csv(
    ROOT / "data/processed/provisional_lyrics_hurdle_modelling_v1/lyric_lexical_features_by_track.csv"
)
lexical_columns = [c for c in lexical.columns if c != "spotify_track_id"]
data = data.merge(lexical, on="spotify_track_id", how="left", validate="one_to_one")

retained_numeric = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_ms", "mode",
    "lex_log_word_count", "lex_log_unique_word_count", "lyric_lexical_diversity",
    "lyric_average_word_length", "lex_log_line_count",
    "lyric_repeated_line_proportion", "lex_log_intensity_punctuation",
]
pearson = data[retained_numeric].corr(method="pearson")
spearman = data[retained_numeric].corr(method="spearman")
frozen_pearson = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/final_predictor_pearson_matrix.csv",
    index_col=0,
)
assert np.allclose(
    pearson.loc[frozen_pearson.index, frozen_pearson.columns].to_numpy(),
    frozen_pearson.to_numpy(), atol=1e-12, equal_nan=True
)

pairs = []
for i, left in enumerate(retained_numeric):
    for right in retained_numeric[i + 1:]:
        pairs.append({
            "variable 1": left, "variable 2": right,
            "Pearson": pearson.loc[left, right],
            "Spearman": spearman.loc[left, right],
            "maximum absolute": max(abs(pearson.loc[left, right]), abs(spearman.loc[left, right])),
        })
strong_pairs = pd.DataFrame(pairs).sort_values("maximum absolute", ascending=False)
display(strong_pairs.head(10).round(3))

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(pearson, dtype=bool), k=1)
sns.heatmap(pearson, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1,
            square=False, linewidths=.3, cbar_kws={"label": "Pearson correlation"}, ax=ax)
ax.set_title("Retained transparent predictors: Pearson correlations")
save_and_show_figure(fig, FIGURE_DIR / "nb03_retained_predictor_correlations.png")

vif = pd.read_csv(ROOT / "data/processed/pre_model_analysis_ready_v1/vif_diagnostic.csv")
final_stability = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/final_coefficient_stability_audit.csv"
)
display(vif.sort_values("vif", ascending=False).head(12).round(2))
display(final_stability.sort_values("vif", ascending=False)[[
    "term", "source_variable", "vif", "coefficient_interpretation_flag"
]].head(10).round(2))


Energy and loudness are strongly related. The three transformed lyric-length quantities are more highly correlated, which makes individual lexical coefficients difficult to interpret in isolation. These diagnostics support cautious interpretation; they are not used as an automatic deletion rule because the predictive question concerns the contribution of prespecified feature groups.

## Frozen grouped and temporal validation

Grouped cross-validation estimates performance for tracks whose lead artist and protected version group were not represented in the held-out fold. Temporal validation asks a different question: whether models trained on 2017–2018 transfer to 2019, with 2020 retained only as a small stress test for Y_52.


In [ ]:
assignments = pd.read_csv(
    ROOT / "data/processed/final_pre_model_specification_v1/provisional_grouped_cv_assignments.csv"
)
temporal52 = pd.read_csv(
    ROOT / "data/processed/final_pre_model_specification_v1/provisional_temporal_validation_52w.csv"
)
temporal26 = pd.read_csv(
    ROOT / "data/processed/final_pre_model_specification_v1/provisional_temporal_validation_26w.csv"
)
crossing = assignments[[
    "first_artist_crosses_folds", "exact_lyric_group_crosses_folds",
    "related_version_group_crosses_folds"
]].sum()
validation_summary = pd.DataFrame([
    ("grouped tracks", len(assignments)),
    ("protected components", assignments.primary_group_component.nunique()),
    ("distinct first credited artists", assignments.first_credited_artist_id.nunique()),
    ("first-artist crossings", crossing["first_artist_crosses_folds"]),
    ("exact-lyric crossings", crossing["exact_lyric_group_crosses_folds"]),
    ("supported-version crossings", crossing["related_version_group_crosses_folds"]),
    ("tracks with cross-fold collaborator overlap", assignments.has_any_credited_artist_cross_fold_overlap.sum()),
], columns=["validation property", "value"])
display(validation_summary)
display(assignments.fold.value_counts().sort_index().rename("tracks").to_frame())
display(pd.concat([
    temporal52.temporal_partition.value_counts().rename("Y_52"),
    temporal26.temporal_partition.value_counts().rename("Y_26"),
], axis=1).fillna(0).astype(int))
assert len(assignments) == 1014
assert assignments.primary_group_component.nunique() == 276
assert crossing.sum() == 0


## Recomputed year-controlled F1 and F3 benchmarks

F1_YEAR and F3_YEAR use the same Poisson estimator and frozen protected folds as their historical counterparts. The only design-matrix addition is the pre-specified categorical entry-year control. The saved targeted-update predictions are used solely to verify the independent notebook recomputation.


In [ ]:
import run_supervisor_confirmed_update as supervisor_update

model_data, frozen_assignments, frozen_temporal = supervisor_update.load_model_data()
F1_YEAR_VARIABLES = supervisor_update.MODEL_VARIABLES["F1_YEAR"]
F3_YEAR_VARIABLES = supervisor_update.MODEL_VARIABLES["F3_YEAR"]
assert blocked_predictor_hits(F1_YEAR_VARIABLES + F3_YEAR_VARIABLES) == []
assert "chart_entry_year" in F1_YEAR_VARIABLES and "chart_entry_year" in F3_YEAR_VARIABLES

prediction_parts = []
for fold in sorted(model_data.fold.unique()):
    train = model_data[model_data.fold.ne(fold)].copy()
    heldout = model_data[model_data.fold.eq(fold)].copy()
    for model_id, variables in [
        ("F1_YEAR", F1_YEAR_VARIABLES),
        ("F3_YEAR", F3_YEAR_VARIABLES),
    ]:
        prediction, _, fitted_model, warning_text = supervisor_update.fit_count(
            train, heldout, variables
        )
        if warning_text:
            warnings.warn(warning_text)
        prediction_parts.append(pd.DataFrame({
            "spotify_track_id": heldout.spotify_track_id.to_numpy(),
            "fold": fold,
            "model_id": model_id,
            "y_observed": heldout.Y_52.to_numpy(),
            "y_pred_raw": prediction,
        }))

recomputed_predictions = pd.concat(prediction_parts, ignore_index=True)
assert not recomputed_predictions.duplicated(["spotify_track_id", "model_id"]).any()
recomputed_metrics = pd.DataFrame([
    {"model_id": model_id, **regression_metrics(part.y_observed, part.y_pred_raw)}
    for model_id, part in recomputed_predictions.groupby("model_id")
])

# Reconciliation uses the independently saved OOF predictions from the targeted update.
saved_predictions = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/year_controlled_grouped_cv_predictions.csv"
)
saved_predictions = saved_predictions[saved_predictions.model_id.isin(["F1_YEAR", "F3_YEAR"])]
aligned = recomputed_predictions.merge(
    saved_predictions[["spotify_track_id", "model_id", "fold", "y_observed", "y_pred_raw"]]
        .rename(columns={"y_pred_raw": "saved_prediction", "y_observed": "saved_outcome"}),
    on=["spotify_track_id", "model_id", "fold"], validate="one_to_one"
)
assert len(aligned) == 2 * 1014
assert np.allclose(aligned.y_observed, aligned.saved_outcome, atol=0)
assert np.allclose(aligned.y_pred_raw, aligned.saved_prediction, atol=1e-10)
display(recomputed_metrics.round(3))

encoding = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/entry_year_encoding_audit.csv"
)
display(encoding)


## Final year-controlled sequence and historical comparison

The canonical calendar implementation treats entry year as categorical. Within grouped cross-validation, the encoder is fitted on each training fold. In temporal validation, 2019 and 2020 are unseen levels relative to 2017–2018 training and are mapped to the fitted reference vector through `handle_unknown='ignore'`. This avoids hidden future-year information, although it means the temporal test cannot estimate a separate effect for a year absent from training.


In [ ]:
year_metrics = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/year_controlled_grouped_cv_metrics.csv"
)
history = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/historical_vs_year_controlled_comparison.csv"
)
display(year_metrics[year_metrics.model_id.isin(["F1_YEAR", "F3_YEAR", "LEX_YEAR", "H3_YEAR"])][[
    "model_id", "n", "mae", "rmse", "mean_prediction_bias", "outside_bound_n"
]].round(3))
display(history[[
    "historical_model_id", "year_controlled_model_id", "historical_mae", "year_controlled_mae",
    "mae_change_year_minus_historical", "historical_rmse", "year_controlled_rmse",
    "rmse_change_year_minus_historical",
]].round(4))

plot_data = year_metrics[
    year_metrics.model_id.isin(["F1_YEAR", "F3_YEAR", "LEX_YEAR", "H3_YEAR"])
].melt(id_vars="model_id", value_vars=["mae", "rmse"], var_name="metric", value_name="error")
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=plot_data, x="model_id", y="error", hue="metric",
            palette=["#4C78A8", "#F58518"], ax=ax)
ax.set(title="Supervisor-confirmed year-controlled model sequence",
       xlabel="Model", ylabel="Prediction error (chart weeks)")
ax.legend(title="", labels=["MAE", "RMSE"])
save_and_show_figure(fig, FIGURE_DIR / "nb03_year_controlled_sequence.png")


Entry year changes the grouped prediction errors only by a few hundredths of a week. X therefore remains the dominant predictor block, while the standard audio descriptors add little beyond it. The year-controlled sequence is now the thesis-facing principal specification; the pre-year results remain preserved historical evidence.

## Notebook 03 conclusions

Predictor eligibility continues to be determined by timing and substantive role rather than performance-driven selection. The protected folds are unchanged, L and Z remain excluded, and the categorical entry-year control is applied consistently without adding release age or reopening the earlier calendar comparison.


In [ ]:
metric_index = recomputed_metrics.set_index("model_id")
record_values(RECORD_DIR / "nb03_values.json", "03_feature_selection_and_benchmarks", {
    "protected_components": frozen_assignments.primary_group_component.nunique(),
    "f1_year_mae": metric_index.loc["F1_YEAR", "mae"],
    "f1_year_rmse": metric_index.loc["F1_YEAR", "rmse"],
    "f3_year_mae": metric_index.loc["F3_YEAR", "mae"],
    "f3_year_rmse": metric_index.loc["F3_YEAR", "rmse"],
    "maximum_saved_prediction_difference": float((aligned.y_pred_raw - aligned.saved_prediction).abs().max()),
    "energy_loudness_pearson": pearson.loc["energy", "loudness"],
})
print("Notebook 03 year-controlled reconciliation record saved.")
